In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load standard built-in dataset
df_raw = sns.load_dataset('titanic')

# Select key columns for cleaning demo
df = df_raw[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()

# Inject deliberate data quality issues (Messy Data Generation)
np.random.seed(42)

# 1. Inconsistent Text Formatting in 'sex' and 'embarked'
df['sex'] = df['sex'].map({'male': 'male', 'female': 'female'})
df.loc[::10, 'sex'] = 'MALE'
df.loc[::15, 'sex'] = 'M'
df.loc[::20, 'sex'] = 'female '

# 2. Incorrect Data Types & String representation for IDs/Dates
df['passenger_id'] = [f"ID_{1000 + i}" for i in range(len(df))]
df['joining_date'] = pd.date_range(start='2022-01-01', periods=len(df), freq='D').astype(str)
df.loc[::25, 'joining_date'] = df['joining_date'].str.replace('-', '/') # Inconsistent date formats

# 3. Outliers in Numeric Columns
df.loc[5, 'fare'] = 1200.0  # extreme fare outlier
df.loc[12, 'age'] = 150.0   # invalid age outlier

# 4. Duplicate Rows
df = pd.concat([df, df.iloc[[10, 25, 50, 100]]], ignore_index=True)

print("Messy Dataset successfully generated! Shape:", df.shape)
df.head()

Messy Dataset successfully generated! Shape: (895, 10)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,passenger_id,joining_date
0,0,3,female,22.0,1,0,7.2500,S,ID_1000,2022/01/01
1,1,1,female,38.0,1,0,71.2833,C,ID_1001,2022-01-02
2,1,3,female,26.0,0,0,7.9250,S,ID_1002,2022-01-03
3,1,1,female,35.0,1,0,53.1000,S,ID_1003,2022-01-04
4,0,3,male,35.0,0,0,8.0500,S,ID_1004,2022-01-05


In [3]:
# Initial Data Quality Audit
print("=== DATA QUALITY REPORT (BEFORE CLEANING) ===")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print(f"\nDuplicate Rows Count: {df.duplicated().sum()}")

# Detailed Column Breakdown
quality_report = pd.DataFrame({
    'Data Type': df.dtypes,
    'Null Count': df.isnull().sum(),
    'Null Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})

# Save initial stats for final before/after comparison
before_stats = {
    'Total Rows': df.shape[0],
    'Duplicate Rows': df.duplicated().sum(),
    'Total Null Values': df.isnull().sum().sum(),
    'Incorrect Dtypes': 3 # age (float with anomalies), joining_date (object), passenger_id (object)
}

quality_report

=== DATA QUALITY REPORT (BEFORE CLEANING) ===
Total Rows: 895
Total Columns: 10

Duplicate Rows Count: 4


,Data Type,Null Count,Null Percentage (%),Unique Values
survived,int64,0,0.00,2
pclass,int64,0,0.00,3
sex,object,0,0.00,5
age,float64,177,19.78,89
sibsp,int64,0,0.00,7
parch,int64,0,0.00,7
fare,float64,0,0.00,248
embarked,object,2,0.22,3
passenger_id,object,0,0.00,891
joining_date,object,0,0.00,891


In [5]:
# Identify and remove duplicate rows
duplicates_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print(f"Successfully identified and removed {duplicates_count} duplicate rows.")
print(f"New Dataset Shape: {df.shape}")

Successfully identified and removed 4 duplicate rows.
New Dataset Shape: (891, 10)


In [7]:
# 1. Standardize 'sex' column
df['sex'] = df['sex'].astype(str).str.strip().str.lower()
gender_map = {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}
df['sex'] = df['sex'].map(gender_map)

# 2. Standardize Date column
df['joining_date'] = pd.to_datetime(df['joining_date'], format='mixed')

# Verify Standardization
print("Unique 'sex' values:", df['sex'].unique())
print("'joining_date' dtype:", df['joining_date'].dtype)

Unique 'sex' values: ['Female' 'Male']
'joining_date' dtype: datetime64[ns]


In [9]:
# Imputation Strategy
# 1. 'age': Impute using Median (Robust against remaining outliers)
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)

# 2. 'embarked': Impute using Mode (Most frequent categorical value)
embarked_mode = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(embarked_mode)

print("Remaining Null Values per Column:")
print(df.isnull().sum())

Remaining Null Values per Column:
survived        0
pclass          0
sex             0
age             0
sibsp           0
parch           0
fare            0
embarked        0
passenger_id    0
joining_date    0
dtype: int64


In [11]:
# Function to cap outliers using Interquartile Range (IQR)
def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = dataframe[(dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)]
    print(f"Column '{column}': Found {len(outliers)} outliers using IQR.")
    
    # Capping (Winsorization)
    dataframe[column] = np.where(dataframe[column] > upper_bound, upper_bound, dataframe[column])
    dataframe[column] = np.where(dataframe[column] < lower_bound, lower_bound, dataframe[column])

# Apply Capping on 'fare' and 'age'
cap_outliers_iqr(df, 'fare')
cap_outliers_iqr(df, 'age')

Column 'fare': Found 117 outliers using IQR.
Column 'age': Found 67 outliers using IQR.


In [13]:
# Type conversions
df['passenger_id'] = df['passenger_id'].astype(str)
df['survived'] = df['survived'].astype(int)
df['pclass'] = df['pclass'].astype(int)
df['age'] = df['age'].round(1)

print("Final Data Types:")
print(df.dtypes)

Final Data Types:
survived                 int32
pclass                   int32
sex                     object
age                    float64
sibsp                    int64
parch                    int64
fare                   float64
embarked                object
passenger_id            object
joining_date    datetime64[ns]
dtype: object


In [15]:
# Compile final comparison stats
after_stats = {
    'Total Rows': df.shape[0],
    'Duplicate Rows': df.duplicated().sum(),
    'Total Null Values': df.isnull().sum().sum(),
    'Incorrect Dtypes': 0
}

summary_df = pd.DataFrame([before_stats, after_stats], index=['Before Cleaning', 'After Cleaning'])
display(summary_df)

,Total Rows,Duplicate Rows,Total Null Values,Incorrect Dtypes
Before Cleaning,895,4,179,3
After Cleaning,891,0,0,0


In [17]:
# Export clean dataset to CSV
output_path = 'cleaned_titanic_dataset.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved successfully to '{output_path}'!")

Cleaned dataset saved successfully to 'cleaned_titanic_dataset.csv'!
